# Transformer Training (OPUS-100)
This notebook walks through training the Transformer model on the Arabic-English OPUS-100 dataset. We will use the `CharTokenizer` for simplicity.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import sys
import os
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR

# Add the root directory to path
sys.path.insert(0, '..')

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

set_seed(42)
print("Seeds set for reproducibility!")

## 1. Load Data & Train Tokenizers

In [2]:
from transformer.data import load_opus100, get_dataloaders
from tokenizers.efficient_bpe import EfficientBPETokenizer

# We'll use a small subset for fast iteration in the notebook
SUBSET_SIZE = 10000

print('Loading data...')
corpus = load_opus100('train', subset_size=SUBSET_SIZE)

# EfficientBPETokenizer takes a list of strings
print('Training Arabic BPE tokenizer...')
ar_texts = [p['ar'] for p in corpus]
src_tokenizer = EfficientBPETokenizer()
src_tokenizer.train(ar_texts, target_vocab_size=4000)

print('Training English BPE tokenizer...')
en_texts = [p['en'] for p in corpus]
tgt_tokenizer = EfficientBPETokenizer()
tgt_tokenizer.train(en_texts, target_vocab_size=4000)

print(f"Source Vocab Size: {src_tokenizer.vocab_size}")
print(f"Target Vocab Size: {tgt_tokenizer.vocab_size}")

Loading data...


ModuleNotFoundError: No module named 'datasets'

## 2. Configuration & Dataloaders

In [ ]:
from transformer.config import TransformerConfig

config = TransformerConfig(
    src_vocab_size=src_tokenizer.vocab_size,
    tgt_vocab_size=tgt_tokenizer.vocab_size,
    d_model=128,
    n_heads=4,
    d_ff=512,
    n_encoder_layers=2,
    n_decoder_layers=2,
    max_seq_len=128,
    batch_size=32,
    epochs=3,
    lr=3e-4
)

train_loader, val_loader = get_dataloaders(
    src_tokenizer, tgt_tokenizer,
    max_len=config.max_seq_len,
    batch_size=config.batch_size,
    subset_size=SUBSET_SIZE
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 3. Setup Model, Optimizer & Scheduler

In [ ]:
from transformer.model import Transformer
from transformer.train import get_lr_scheduler

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

model = Transformer(config).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

optimizer = Adam(model.parameters(), lr=config.lr, betas=(0.9, 0.98), eps=1e-9)
scheduler = get_lr_scheduler(optimizer, config.d_model, config.warmup_steps)
criterion = nn.CrossEntropyLoss(ignore_index=0) # Assuming 0 is PAD ID

## 4. Training Loop

In [ ]:
import matplotlib.pyplot as plt

def train_epoch(model, loader, optimizer, scheduler, criterion, device, epoch):
    model.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(loader):
        src = batch['src_ids'].to(device)
        tgt = batch['tgt_ids'].to(device)
        
        tgt_input = tgt[:, :-1]
        tgt_target = tgt[:, 1:]
        
        src_mask = Transformer.generate_padding_mask(src, pad_id=0)
        tgt_mask = Transformer.generate_causal_mask(tgt_input.size(1)).to(device)
        
        logits = model(src, tgt_input, src_mask, tgt_mask)
        loss = criterion(logits.reshape(-1, config.tgt_vocab_size), tgt_target.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"Epoch {epoch} | Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}")
            
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            src = batch['src_ids'].to(device)
            tgt = batch['tgt_ids'].to(device)
            tgt_input = tgt[:, :-1]
            tgt_target = tgt[:, 1:]
            
            src_mask = Transformer.generate_padding_mask(src, pad_id=0)
            tgt_mask = Transformer.generate_causal_mask(tgt_input.size(1)).to(device)
            
            logits = model(src, tgt_input, src_mask, tgt_mask)
            loss = criterion(logits.reshape(-1, config.tgt_vocab_size), tgt_target.reshape(-1))
            total_loss += loss.item()
            
    return total_loss / len(loader)

# Track metrics
train_history = []
val_history = []

for epoch in range(1, config.epochs + 1):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device, epoch)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    train_history.append(train_loss)
    val_history.append(val_loss)
    
    print(f"\n--- Epoch {epoch} Summary ---")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}\n")

# Plot metrics
plt.figure(figsize=(10, 5))
plt.plot(range(1, config.epochs + 1), train_history, label='Train Loss', marker='o')
plt.plot(range(1, config.epochs + 1), val_history, label='Validation Loss', marker='o')
plt.xlabel('Epochs')
plt.ylabel('Cross Entropy Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

## 5. Inference / Translation
Let's write a small greedy decoding loop to test the translation.

In [ ]:
def translate(text, model, src_tokenizer, tgt_tokenizer, max_len=50):
    model.eval()
    
    # Tokenize input
    src_tokens = src_tokenizer.encode(text)
    src_tensor = torch.tensor([[2] + src_tokens + [3]]).to(device)  # [BOS, ..., EOS]
    src_mask = Transformer.generate_padding_mask(src_tensor, pad_id=0)
    
    with torch.no_grad():
        enc_output = model.encoder(src_tensor, src_mask)
        
        # Start with just BOS token for target
        tgt_tokens = [2]
        
        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_tokens]).to(device)
            tgt_mask = Transformer.generate_causal_mask(tgt_tensor.size(1)).to(device)
            
            dec_output = model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask)
            logits = model.output_proj(dec_output)
            
            # Get the token with highest probability for the last position
            next_token = logits[0, -1, :].argmax().item()
            
            if next_token == 3:  # EOS token
                break
                
            tgt_tokens.append(next_token)
            
    # Decode output tokens
    # Note: we skip the initial BOS token
    return tgt_tokenizer.decode(tgt_tokens[1:])

# Test it on a sample from the validation set
sample_ar = corpus[10]['ar']
sample_en_true = corpus[10]['en']
pred_en = translate(sample_ar, model, src_tokenizer, tgt_tokenizer)

print(f"Arabic:  {sample_ar}")
print(f"English: {sample_en_true}")
print(f"Pred:    {pred_en}")

## 6. Attention Visualization
Let's plot the cross-attention weights of the last decoder block to see what Arabic words the model was looking at while generating English words.

In [ ]:
import seaborn as sns

def plot_attention_weights(model, text, src_tokenizer, tgt_tokenizer, max_len=50):
    model.eval()
    
    src_tokens = src_tokenizer.encode(text)
    src_tensor = torch.tensor([[2] + src_tokens + [3]]).to(device)
    src_mask = Transformer.generate_padding_mask(src_tensor, pad_id=0)
    
    with torch.no_grad():
        enc_output = model.encoder(src_tensor, src_mask)
        tgt_tokens = [2]
        
        for _ in range(max_len):
            tgt_tensor = torch.tensor([tgt_tokens]).to(device)
            tgt_mask = Transformer.generate_causal_mask(tgt_tensor.size(1)).to(device)
            
            dec_output = model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask)
            logits = model.output_proj(dec_output)
            
            next_token = logits[0, -1, :].argmax().item()
            if next_token == 3:
                break
            tgt_tokens.append(next_token)
            
    # Extract attention weights from the last decoder layer's cross-attention module
    # Shape: (batch, n_heads, tgt_len, src_len)
    attn_weights = model.decoder.blocks[-1].cross_attn.attention.attn_weights[0].cpu().numpy()
    
    # We will plot the average attention across all heads
    avg_attn = attn_weights.mean(axis=0)
    
    # Get string tokens for labels
    # We strip the initial BOS from target for visualization to align with generated words
    ar_labels = [src_tokenizer.decode([t]) for t in src_tensor[0].tolist()]
    en_labels = [tgt_tokenizer.decode([t]) for t in tgt_tokens[1:]]
    
    # We trim the attention map to only show the generated target tokens (excluding BOS)
    avg_attn = avg_attn[1:, :]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(avg_attn, xticklabels=ar_labels, yticklabels=en_labels, cmap='viridis', 
                cbar_kws={'label': 'Attention Weight'})
    plt.xlabel('Source Arabic Tokens')
    plt.ylabel('Generated English Tokens')
    plt.title('Cross-Attention Heatmap (Averaged across heads)')
    plt.tight_layout()
    plt.show()

# Visualize
plot_attention_weights(model, sample_ar, src_tokenizer, tgt_tokenizer)